In [1]:
map(x -> x^2, 1:8)

8-element Array{Int64,1}:
  1
  4
  9
 16
 25
 36
 49
 64

In [2]:
A = 1:5
B = [1  2
     3  4
     5  6
     7  8
     9 10]
broadcast(+, A, B) # adds A to ea col of B

5×2 Array{Int64,2}:
  2   3
  5   6
  8   9
 11  12
 14  15

In [8]:
B = [2; 3; 4; 5; 6]
A .* B # elem wise mult

5-element Array{Int64,1}:
  2
  6
 12
 20
 30

In [9]:
C = [3; 4; 5; 2; 1]
A .* B .* C

5-element Array{Int64,1}:
  6
 24
 60
 40
 30

In [5]:
function broadcastmult(x, y)
    output = similar(x) 
    for i in eachindex(x)
        output[i] = x[i] * y[i]
    end
    output
end

broadcastmult (generic function with 1 method)

In [6]:
function broadcastmult(x, y, z)
    output = similar(x)
    for i in eachindex(x)
        output[i] = x[i] * y[i] * z[i]
    end
    output
end

broadcastmult (generic function with 2 methods)

In [11]:
broadcast((x, y, z) -> x * y * z, A, B, C)

5-element Array{Int64,1}:
  6
 24
 60
 40
 30

In [12]:
A .* B .* sin.(C)

5-element Array{Float64,1}:
   0.2822400161197344
  -4.540814971847569 
 -11.507091295957661 
  18.185948536513635 
  25.244129544236895 

In [13]:
D = similar(C)

5-element Array{Int64,1}:
          0
          0
          0
          0
 4496293888

In [20]:
using BenchmarkTools
@btime D .= A .* B .* C

┌ Info: Precompiling BenchmarkTools [6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf]
└ @ Base loading.jl:1186
┌ Warning: Module JSON with build ID 77344327201035 is missing from the cache.
│ This may mean JSON [682c06a0-de6a-54ab-a142-c8b1cf79cde6] does not support precompilation but is imported by a module that does.
└ @ Base loading.jl:941


DimensionMismatch: DimensionMismatch("arrays could not be broadcast to a common size")

In [15]:
A = rand(4, 4)
A[1:3, 1:3]

3×3 Array{Float64,2}:
 0.412294  0.0918523  0.0429264
 0.33579   0.578483   0.154925 
 0.831048  0.685196   0.677634 

In [16]:
@time A[1:3, 1:3]

  0.000020 seconds (7 allocations: 384 bytes)


3×3 Array{Float64,2}:
 0.412294  0.0918523  0.0429264
 0.33579   0.578483   0.154925 
 0.831048  0.685196   0.677634 

In [17]:
a = [1; 3; 5]
@time b = a

a[2] = 10
a

@time c = copy(a)

  0.000001 seconds (4 allocations: 192 bytes)
  0.001327 seconds (28 allocations: 1.828 KiB)


3-element Array{Int64,1}:
  1
 10
  5

In [18]:
A = rand(4, 4)

4×4 Array{Float64,2}:
 0.109986   0.519266  0.682253  0.979547
 0.151156   0.268899  0.943713  0.140976
 0.658812   0.847299  0.931729  0.620602
 0.0356013  0.779212  0.160375  0.43775 

In [19]:
function testloops()
    b = rand(1000, 1000)
    c = 0 # prevents compiler from loop optimization
    @time for i in 1:1000, j in 1:1000
        c += b[i, j]
    end
    @time for j in 1:1000, i in 1:1000
        c += b[i, j]
    end
    bidx = eachindex(b)
    @time for i in bidx
        c += b[i]
    end
end

testloops()

  0.000899 seconds
  0.000578 seconds
  0.000286 seconds


In [21]:
println(A)
B = A[1:3, 1:3] # copy
B[1, 1] = 100
println(A)

[0.109986 0.519266 0.682253 0.979547; 0.151156 0.268899 0.943713 0.140976; 0.658812 0.847299 0.931729 0.620602; 0.0356013 0.779212 0.160375 0.43775]
[0.109986 0.519266 0.682253 0.979547; 0.151156 0.268899 0.943713 0.140976; 0.658812 0.847299 0.931729 0.620602; 0.0356013 0.779212 0.160375 0.43775]


In [22]:
B = view(A, 1:3, 1:3) # NOT a copy
B[1, 1] = 777
println(A)

[777.0 0.519266 0.682253 0.979547; 0.151156 0.268899 0.943713 0.140976; 0.658812 0.847299 0.931729 0.620602; 0.0356013 0.779212 0.160375 0.43775]


In [23]:
C = vec(A)
println(C)
C = reshape(A, 8, 2)
C

[777.0, 0.151156, 0.658812, 0.0356013, 0.519266, 0.268899, 0.847299, 0.779212, 0.682253, 0.943713, 0.931729, 0.160375, 0.979547, 0.140976, 0.620602, 0.43775]


8×2 Array{Float64,2}:
 777.0        0.682253
   0.151156   0.943713
   0.658812   0.931729
   0.0356013  0.160375
   0.519266   0.979547
   0.268899   0.140976
   0.847299   0.620602
   0.779212   0.43775 

In [24]:
A = rand(4, 4)
B = rand(4, 4)
C = A * B  # mat mult
D = A .* B # Elem wise mult
C - D      # ≠ 0

4×4 Array{Float64,2}:
 0.313384  0.109636  0.188276  0.160546
 0.909463  1.27284   1.17725   0.37276 
 0.78427   0.759256  0.541443  0.259849
 0.650291  0.988279  0.705637  0.277077

In [27]:
b = 1:4
x = A \ b # solve x for Ax=b

4-element Array{Float64,1}:
  4.908494743570489  
  3.3965360120754227 
 -3.3489085148794593 
  0.48743743220322194

In [28]:
A * x

4-element Array{Float64,1}:
 1.0               
 2.0               
 3.0000000000000004
 4.000000000000001 

In [29]:
i = 1
C[i, :] .= view(A, i, :) .* view(B, i, :)

4-element view(::Array{Float64,2}, 1, :) with eltype Float64:
 0.00936528299412767 
 0.1659133432285296  
 0.022487449997951662
 0.046712005851696395

In [30]:
using SparseArrays

In [32]:
A = sparse([1; 2; 3], [2; 2; 1], [3; 4; 5]) # (rowinds, colinds, vals)

3×2 SparseMatrixCSC{Int64,Int64} with 3 stored entries:
  [3, 1]  =  5
  [1, 2]  =  3
  [2, 2]  =  4

In [34]:
Array(A)

3×2 Array{Int64,2}:
 0  3
 0  4
 5  0

In [35]:
using LinearAlgebra

In [36]:
A = Tridiagonal(2:5, 1:5, 1:4)

5×5 Tridiagonal{Int64,UnitRange{Int64}}:
 1  1  ⋅  ⋅  ⋅
 2  2  2  ⋅  ⋅
 ⋅  3  3  3  ⋅
 ⋅  ⋅  4  4  4
 ⋅  ⋅  ⋅  5  5

In [37]:
fieldnames(Tridiagonal)

(:dl, :d, :du, :du2)

In [40]:
A.d

1:5

In [41]:
A * rand(5, 5)

5×5 Array{Float64,2}:
 1.17132  1.59042  0.991626  0.740845  0.495087
 3.72128  4.26741  3.12203   3.42385   2.54567 
 7.00856  7.04009  4.04377   4.56811   6.27025 
 7.86786  9.4505   4.19945   5.97808   8.62893 
 6.38823  9.0967   2.40237   2.6172    6.89742 

In [43]:
A = rand(5, 5)
λ = 2
A - λ*I

5×5 Array{Float64,2}:
 -1.34738     0.473168   0.202842    0.871107   0.524751
  0.755561   -1.10568    0.917311    0.567339   0.682746
  0.0164498   0.163968  -1.24045     0.42517    0.772165
  0.942143    0.480688   0.0658577  -1.33245    0.796126
  0.134168    0.595134   0.93931     0.39423   -1.07787 

In [44]:
b = 1:5
A = rand(5, 5)
Q, R = qr(A) # QR Factorization
println(A \ b)
println(inv(R) * Q' * b)

[-0.763649, 0.547493, -0.861847, 1.25401, 5.86599]
[-0.763649, 0.547493, -0.861847, 1.25401, 5.86599]


In [45]:
q = qr(A)

LinearAlgebra.QRCompactWY{Float64,Array{Float64,2}}
Q factor:
5×5 LinearAlgebra.QRCompactWYQ{Float64,Array{Float64,2}}:
 -0.139249     0.958821    0.0183289  -0.123578  -0.213692
 -0.72573     -0.206678    0.475855   -0.417827  -0.171994
 -0.593924    -0.0695464  -0.669623    0.388684  -0.207239
 -0.00487085  -0.0077259  -0.545543   -0.757009   0.359495
 -0.31805      0.181796    0.164966    0.293277   0.867507
R factor:
5×5 Array{Float64,2}:
 -1.13218  -1.38581   -0.620182   -0.796028  -0.788332 
  0.0       0.746699   0.0890732   0.277081   0.0912708
  0.0       0.0       -0.853107   -0.100679  -0.512303 
  0.0       0.0        0.0        -0.408456  -0.143646 
  0.0       0.0        0.0         0.0        0.78352  

In [46]:
q \ b

5-element Array{Float64,1}:
 -0.7636487010731209
  0.5474931467255056
 -0.8618466478905672
  1.254013686615931 
  5.865991170791416 

In [47]:
rand(5)

5-element Array{Float64,1}:
 0.8064193672095417 
 0.800432243696032  
 0.36125488825020513
 0.7385336164535652 
 0.787126771418607  

In [48]:
randn(5, 5) # normal(0, 1) ...?

5×5 Array{Float64,2}:
  0.112963  -1.9049    -1.57193    0.545592   0.05915 
 -0.680021   2.11561    1.57255   -0.604203   0.169128
  1.63758   -1.36236   -0.213345   0.946862   1.78325 
 -0.49885   -0.340061  -0.115011  -0.190168  -0.239036
  0.148881   0.536465  -0.208219  -0.379      1.26096 

In [49]:
rand(5, 5) # U(0, 1)

5×5 Array{Float64,2}:
 0.584163  0.691196  0.702644  0.331601  0.42785 
 0.888931  0.260203  0.120103  0.620047  0.638463
 0.444669  0.829447  0.219866  0.129741  0.734949
 0.537677  0.638171  0.281367  0.259881  0.748654
 0.540451  0.443544  0.239987  0.276901  0.268043

In [50]:
a = [1 2; 3 4]
randn(size(a))

2×2 Array{Float64,2}:
 1.35262   0.606242
 0.133546  0.17326 